In [ ]:
import numpy as np
import pandas as pd
import sys
import os
import pickle
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import json
sys.path.append("../../../src/experiment_util")
import experiment_utils

In [ ]:
with open("../../../dir_paths.json") as f:
    config = json.load(f)
    dataframes_path = config["dataframes_path"]
    project_output_path = config["project_output_path"]

gail_policies = pd.read_pickle(dataframes_path + "/first_n_gail.pkl")


In [ ]:
misclassified_users_rf_no_agree_all_subs_dict = {}
misclassified_users_xgb_no_agree_all_subs_dict = {}


gail_policies_results = []

for n in gail_policies.n.unique():
    running_confusion_matrix_rf = np.zeros((2, 2), dtype=int)
    running_confusion_matrix_xgb = np.zeros((2, 2), dtype=int)

    f1_list_rf = []
    f1_list_xgb = []

    misclassified_users_rf_all_subs = defaultdict(int)
    misclassified_users_xgb_all_subs = defaultdict(int)
    for run in gail_policies.run.unique():
        
        print(n,"run:",run)
        seed = 100 + run

        df_russian_sampled = gail_policies[(gail_policies.run == run) & (gail_policies.n == n) & (gail_policies.russian == 1)].copy()
        df_sampled = gail_policies[(gail_policies.run == run) & (gail_policies.n == n) & (gail_policies.russian == 0)].copy()

        df_russian_sampled["feature_col"] = df_russian_sampled['policy'].apply(lambda x: x.flatten())
        df_sampled["feature_col"] = df_sampled['policy'].apply(lambda x: x.flatten())

        confusion_matrix_rf, confusion_matrix_xgb, overall_rf_f1, overall_xgb_f1, misclassified_users_rf, misclassified_users_xgb = experiment_utils.run_experiment(df_russian_sampled,df_sampled,seed = seed)

        f1_list_rf.append(overall_rf_f1)
        f1_list_xgb.append(overall_xgb_f1)
        running_confusion_matrix_rf += confusion_matrix_rf
        running_confusion_matrix_xgb += confusion_matrix_xgb
        for user in misclassified_users_rf:
            misclassified_users_rf_all_subs[user] += misclassified_users_rf[user]
        for user in misclassified_users_xgb:
            misclassified_users_xgb_all_subs[user] += misclassified_users_xgb[user]

    print("n", n)
    print("Final Confusion Matrix (Random Forest):\n", running_confusion_matrix_rf)    
    f1 = np.mean(f1_list_rf)
    print(f"Mean F1 Score (Random Forest): {f1:.2f}\n")

    print("Final Confusion Matrix (XGBoost):\n", running_confusion_matrix_xgb)        
    f1 = np.mean(f1_list_xgb)
    print(f"Mean F1 Score (XGBoost): {f1:.2f}\n")


    results = {
        "n":n,
        "f1_list_rf":f1_list_rf,
        "f1_list_xgb":f1_list_xgb,
        "running_confusion_matrix_rf":running_confusion_matrix_rf,
        "running_confusion_matrix_xgb":running_confusion_matrix_xgb,
        "misclassified_users_rf_all_subs":misclassified_users_rf_all_subs,
        "misclassified_users_xgb_all_subs":misclassified_users_xgb_all_subs
    }

    misclassified_users_rf_no_agree_all_subs_dict[n] = misclassified_users_rf_all_subs
    misclassified_users_xgb_no_agree_all_subs_dict[n] = misclassified_users_xgb_all_subs

    gail_policies_results.append(results)

output_dir = project_output_path + "/early_detection"
os.makedirs(output_dir, exist_ok=True)
with open(output_dir +'/gail_policies_results.pkl', 'wb') as f:
    pickle.dump(gail_policies_results, f)

